In [3]:
import os

# 실제 경로 찾기
for root, dirs, files in os.walk("/"):
    for f in files:
        if f == "adapter_config.json":
            print(os.path.join(root, f))

/home/ubuntu/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct/.no_exist/aa8e72537993ba99e69dfaafa59ed015b17504d1/adapter_config.json
/home/ubuntu/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct/.no_exist/7ae557604adf67be50417f59c2c2f167def9a775/adapter_config.json
/home/ubuntu/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/.no_exist/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/adapter_config.json
/home/ubuntu/qwen_project/checkpoint-1610/adapter_config.json
/home/ubuntu/qwen_project/final/adapter_config.json
/home/ubuntu/qwen_project/result/qwen_1.5b_ft/final/adapter_config.json
/home/ubuntu/qwen_project/result/qwen_1.5b_ft/checkpoint-306/adapter_config.json
/home/ubuntu/qwen_project/result/qwen_1.5b_ft/checkpoint-612/adapter_config.json
/home/ubuntu/qwen_project/result/qwen_1.5b_ft/checkpoint-918/adapter_config.json
/home/ubuntu/qwen_project/result/qwen_1.5b_adv/checkpoint-1610/adapter_config.json
/home/ubuntu/qwen_project/result/qwen_1.5b_adv/final/adapter_

In [4]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch.nn.functional as F
from scipy.stats import entropy

# ── 1. 경로 설정 ──────────────────────────────────────────
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "/home/ubuntu/qwen_project/result/qwen_1.5b_ft/final"

# ── 2. 모델 로드 ──────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("모델 로드 완료")

# ── 3. 레이블 토큰 ID ─────────────────────────────────────
LABELS = ["support", "deny", "query", "comment"]
label_ids = {lab: tokenizer.encode(lab, add_special_tokens=False)[0] for lab in LABELS}
print("레이블 토큰:", label_ids)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 552.05it/s]


모델 로드 완료
레이블 토큰: {'support': 23362, 'deny': 89963, 'query': 1631, 'comment': 6182}


In [5]:
# ── 4. Confidence 측정 함수 ───────────────────────────────
def get_confidence_scores(prompt: str):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)

    last_logits = outputs.logits[0, -1, :]
    label_logits = torch.tensor([last_logits[label_ids[l]] for l in LABELS])
    probs = F.softmax(label_logits, dim=0).cpu().numpy()

    prob_dict   = dict(zip(LABELS, probs))
    pred        = LABELS[np.argmax(probs)]
    max_prob    = float(np.max(probs))
    entropy_val = float(entropy(probs))
    sorted_p    = np.sort(probs)[::-1]
    margin      = float(sorted_p[0] - sorted_p[1])

    return pred, prob_dict, max_prob, entropy_val, margin

# ── 5. 간단 테스트 ────────────────────────────────────────
test_prompt = """Classify the stance of the reply toward the rumor.
Labels: support, deny, query, comment

Reply: "This is completely fake news."

Answer:"""

pred, probs, conf, ent, margin = get_confidence_scores(test_prompt)
print(f"예측: {pred}")
print(f"확률 분포: { {k: f'{v:.3f}' for k,v in probs.items()} }")
print(f"Confidence: {conf:.3f}")
print(f"Entropy(불확실성): {ent:.3f}")
print(f"Margin(1위-2위 차이): {margin:.3f}")

예측: deny
확률 분포: {'support': '0.005', 'deny': '0.993', 'query': '0.001', 'comment': '0.001'}
Confidence: 0.993
Entropy(불확실성): 0.047
Margin(1위-2위 차이): 0.988


In [7]:
import os
for root, dirs, files in os.walk("/home/ubuntu/qwen_project"):
    for f in files:
        if f.endswith(".json") and "dev" in f:
            print(os.path.join(root, f))

/home/ubuntu/qwen_project/rumoureval-2019-training-data/dev-key.json
/home/ubuntu/qwen_project/rumoureval-2019-training-data/.ipynb_checkpoints/dev-key-checkpoint.json


In [10]:
# 파일 하나 열어서 구조 확인
import json

sample_path = "/home/ubuntu/qwen_project/rumoureval-2019-training-data/twitter-english/illary/769988636754505729/replies/769994652569264128.json"

with open(sample_path, "r") as f:
    sample = json.load(f)

print("키들:", list(sample.keys()))
print("\ntext 있으면:", sample.get("text", "없음"))

키들: ['contributors', 'truncated', 'text', 'is_quote_status', 'in_reply_to_status_id', 'id', 'favorite_count', 'source', 'retweeted', 'coordinates', 'entities', 'in_reply_to_screen_name', 'in_reply_to_user_id', 'retweet_count', 'id_str', 'favorited', 'user', 'geo', 'in_reply_to_user_id_str', 'lang', 'created_at', 'in_reply_to_status_id_str', 'place']

text 있으면: @PrisonPlanet ironically the same number of women Bill has slept with since then as well.


In [12]:
import os
import json
from collections import defaultdict, Counter

BASE = "/home/ubuntu/qwen_project/rumoureval-2019-training-data"

In [13]:
def parse_rumoureval(base_dir, key_files):
    label_map = {}
    for key_file in key_files:
        with open(key_file) as f:
            label_map.update(json.load(f).get('subtaskaenglish', {}))

    records = []

    # Reddit 파싱
    for split, subdir in [('dev', 'reddit-dev-data'), ('train', 'reddit-training-data')]:
        data_dir = os.path.join(base_dir, subdir)
        if not os.path.exists(data_dir):
            continue
        for thread_id in os.listdir(data_dir):
            thread_path = os.path.join(data_dir, thread_id)
            if not os.path.isdir(thread_path):
                continue
            src_dir = os.path.join(thread_path, 'source-tweet')
            if not os.path.exists(src_dir):
                continue
            src_files = os.listdir(src_dir)
            if not src_files:
                continue
            with open(os.path.join(src_dir, src_files[0])) as f:
                src_data = json.load(f)
            src_body = src_data['data']['children'][0]['data']
            src_text = src_body.get('selftext') or src_body.get('title', '')
            src_id   = str(src_body.get('id', thread_id))
            records.append({'reply_id': src_id, 'thread_id': thread_id,
                            'parent_id': None, 'depth': -1, 'text': src_text,
                            'label': label_map.get(thread_id), 'platform': 'reddit',
                            'split': split, 'is_source': True})
            rep_dir = os.path.join(thread_path, 'replies')
            if not os.path.exists(rep_dir):
                continue
            for rep_file in os.listdir(rep_dir):
                with open(os.path.join(rep_dir, rep_file)) as f:
                    rep_data = json.load(f)
                rep_body = rep_data['data']
                rep_id   = str(rep_body.get('id', rep_file.replace('.json', '')))
                records.append({'reply_id': rep_id, 'thread_id': thread_id,
                                'parent_id': str(rep_body.get('parent_id', '').split('_')[-1]),
                                'depth': rep_body.get('depth', 0), 'text': rep_body.get('body', ''),
                                'label': label_map.get(rep_id), 'platform': 'reddit',
                                'split': split, 'is_source': False})

    # Twitter 파싱
    twitter_dir = os.path.join(base_dir, 'twitter-english')
    for event_id in os.listdir(twitter_dir):
        event_path = os.path.join(twitter_dir, event_id)
        if not os.path.isdir(event_path):
            continue
        for thread_id in os.listdir(event_path):
            thread_path = os.path.join(event_path, thread_id)
            if not os.path.isdir(thread_path):
                continue
            src_dir = os.path.join(thread_path, 'source-tweet')
            if not os.path.exists(src_dir):
                continue
            src_files = os.listdir(src_dir)
            if not src_files:
                continue
            with open(os.path.join(src_dir, src_files[0])) as f:
                src_data = json.load(f)
            src_id   = str(src_data.get('id', thread_id))
            src_text = src_data.get('text', '')
            split    = 'dev' if src_id in label_map else 'train'
            records.append({'reply_id': src_id, 'thread_id': thread_id,
                            'parent_id': None, 'depth': -1, 'text': src_text,
                            'label': label_map.get(src_id), 'platform': 'twitter',
                            'split': split, 'is_source': True})
            rep_dir = os.path.join(thread_path, 'replies')
            if not os.path.exists(rep_dir):
                continue
            for rep_file in os.listdir(rep_dir):
                with open(os.path.join(rep_dir, rep_file)) as f:
                    rep_data = json.load(f)
                rep_id    = str(rep_data.get('id', rep_file.replace('.json', '')))
                parent_id = str(rep_data.get('in_reply_to_status_id', ''))
                records.append({'reply_id': rep_id, 'thread_id': thread_id,
                                'parent_id': parent_id, 'depth': 0, 'text': rep_data.get('text', ''),
                                'label': label_map.get(rep_id), 'platform': 'twitter',
                                'split': split, 'is_source': False})
    return records

records = parse_rumoureval(
    base_dir=BASE,
    key_files=[
        f"{BASE}/dev-key.json",
        f"{BASE}/train-key.json"
    ]
)

print(f"전체 레코드 수: {len(records)}")
replies = [r for r in records if not r['is_source'] and r['label']]
print(f"레이블 있는 reply: {len(replies)}")
print(Counter(r['label'] for r in replies))

전체 레코드 수: 6702
레이블 있는 reply: 6337
Counter({'comment': 4689, 'support': 715, 'query': 487, 'deny': 446})


In [14]:
# 셀 3: thread 인덱싱 + dev 데이터 추출
threads      = defaultdict(list)
for r in records:
    threads[r['thread_id']].append(r)

id_to_record = {r['reply_id']: r for r in records}

# dev set만 추출 (레이블 있는 reply)
dev_data = [r for r in records if r['split'] == 'dev' and not r['is_source'] and r['label']]
print(f"dev reply 수: {len(dev_data)}")
print(Counter(r['label'] for r in dev_data))

dev reply 수: 5669
Counter({'comment': 4077, 'support': 706, 'query': 472, 'deny': 414})


In [16]:
# 셀 4: context 조건별 프롬프트 생성 함수
STANCE_KEYWORDS = {
    'support': ['confirmed', 'true', 'real', 'verified', 'correct'],
    'deny':    ['fake', 'false', 'denied', 'hoax', 'debunked'],
}

def get_source_text(target, threads, id_to_record):
    thread = threads[target['thread_id']]
    src = next((r for r in thread if r['is_source']), None)
    return src['text'] if src else None

def get_parent_text(target, id_to_record):
    pid = target.get('parent_id')
    if not pid or pid == target['reply_id']:
        return None
    parent = id_to_record.get(str(pid))
    return parent['text'] if parent and parent['text'] else None

def get_irrelevant_text(target, threads):
    """같은 스레드에서 stance 키워드 없는 comment"""
    thread = threads[target['thread_id']]
    all_kw = STANCE_KEYWORDS['support'] + STANCE_KEYWORDS['deny']
    for r in thread:
        if r['reply_id'] == target['reply_id'] or r['is_source']:
            continue
        if r['label'] == 'comment' and r['text']:
            if not any(kw in r['text'].lower() for kw in all_kw):
                return r['text']
    return None

def get_conflicting_text(target, threads):
    """반대 stance 키워드가 있는 reply"""
    thread = threads[target['thread_id']]
    true_label = target['label']
    opposing = 'deny' if true_label == 'support' else 'support'
    opp_kws = STANCE_KEYWORDS.get(opposing, [])
    for r in thread:
        if r['reply_id'] == target['reply_id'] or r['is_source']:
            continue
        if r['text'] and any(kw in r['text'].lower() for kw in opp_kws):
            return r['text']
    return None

def build_prompt(reply_text, source_text=None, context_texts=None, condition='reply_only'):
    parts = []
    if source_text:
        parts.append(f"Rumor: {source_text}")
    if context_texts:
        for ct in context_texts:
            parts.append(f"Context: {ct}")
    parts.append(f"Reply: {reply_text}")
    parts.append("Classify the stance: support / deny / query / comment\nAnswer:")
    return "\n\n".join(parts)

def build_all_conditions(target, threads, id_to_record):
    reply_text     = target['text']
    source_text    = get_source_text(target, threads, id_to_record)
    parent_text    = get_parent_text(target, id_to_record)
    irrelevant     = get_irrelevant_text(target, threads)
    conflicting    = get_conflicting_text(target, threads)

    useful_ctx     = [t for t in [parent_text] if t]
    mixed_ctx      = [t for t in [parent_text, conflicting] if t]
    lexical_ctx    = [conflicting] if conflicting else None

    conditions = {
        'reply_only':  build_prompt(reply_text),
        'useful':      build_prompt(reply_text, source_text, useful_ctx or None),
        'irrelevant':  build_prompt(reply_text, source_text, [irrelevant] if irrelevant else None),
        'conflicting': build_prompt(reply_text, source_text, [conflicting] if conflicting else None),
        'mixed':       build_prompt(reply_text, source_text, mixed_ctx or None),
        'lexical':     build_prompt(reply_text, None, lexical_ctx),
    }
    return conditions

# 테스트
sample = dev_data[0]
conds  = build_all_conditions(sample, threads, id_to_record)
print(f"reply_id: {sample['reply_id']}, label: {sample['label']}\n")
for cname, prompt in conds.items():
    print(f"=== {cname} ===")
    print(prompt[:200])
    print()

reply_id: cbkixew, label: comment

=== reply_only ===
Reply: 
Fukushima is still in the clean-up process. There are still issues with radiation leakages - now whilst *any* radiation leak is considered serious, technically speaking it is only affecting a 

=== useful ===
Rumor: Fukushima spewing equivalent of 112 Hiroshima-type nuclear bombs worth of radiation every hour, of every day. (This canNOT be true...)

Context: Fukushima spewing equivalent of 112 Hiroshima-ty

=== irrelevant ===
Rumor: Fukushima spewing equivalent of 112 Hiroshima-type nuclear bombs worth of radiation every hour, of every day. (This canNOT be true...)

Context: The iodine has an 8 day halflife the caesium 30 

=== conflicting ===
Rumor: Fukushima spewing equivalent of 112 Hiroshima-type nuclear bombs worth of radiation every hour, of every day. (This canNOT be true...)

Context: The bombs dropped during world war 2 were fairly

=== mixed ===
Rumor: Fukushima spewing equivalent of 112 Hiroshima-type nuclear bomb

In [17]:
# 셀 5: dev 전체에 대해 조건별 confidence 측정
from scipy.stats import entropy
import torch.nn.functional as F
import numpy as np

def get_confidence_scores(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    last_logits = outputs.logits[0, -1, :]
    label_logits = torch.tensor([last_logits[label_ids[l]] for l in LABELS])
    probs = F.softmax(label_logits, dim=0).cpu().numpy()
    prob_dict   = dict(zip(LABELS, probs))
    pred        = LABELS[np.argmax(probs)]
    max_prob    = float(np.max(probs))
    entropy_val = float(entropy(probs))
    sorted_p    = np.sort(probs)[::-1]
    margin      = float(sorted_p[0] - sorted_p[1])
    return pred, prob_dict, max_prob, entropy_val, margin

CONDITION_NAMES = ['reply_only', 'useful', 'irrelevant', 'conflicting', 'mixed', 'lexical']

results = []
for i, target in enumerate(dev_data):
    if i % 50 == 0:
        print(f"{i}/{len(dev_data)} 처리 중...")

    conds = build_all_conditions(target, threads, id_to_record)
    row = {
        'reply_id': target['reply_id'],
        'true_label': target['label'],
    }
    for cname in CONDITION_NAMES:
        prompt = conds[cname]
        pred, probs, conf, ent, margin = get_confidence_scores(prompt)
        row[f'{cname}_pred']    = pred
        row[f'{cname}_conf']    = conf
        row[f'{cname}_entropy'] = ent
        row[f'{cname}_margin']  = margin
        row[f'{cname}_correct'] = int(pred == target['label'])
    results.append(row)

print(f"\n완료! 총 {len(results)}개")

0/5669 처리 중...
50/5669 처리 중...
100/5669 처리 중...
150/5669 처리 중...
200/5669 처리 중...
250/5669 처리 중...
300/5669 처리 중...
350/5669 처리 중...
400/5669 처리 중...
450/5669 처리 중...
500/5669 처리 중...
550/5669 처리 중...
600/5669 처리 중...
650/5669 처리 중...
700/5669 처리 중...
750/5669 처리 중...
800/5669 처리 중...
850/5669 처리 중...
900/5669 처리 중...
950/5669 처리 중...
1000/5669 처리 중...
1050/5669 처리 중...
1100/5669 처리 중...
1150/5669 처리 중...
1200/5669 처리 중...
1250/5669 처리 중...
1300/5669 처리 중...
1350/5669 처리 중...
1400/5669 처리 중...
1450/5669 처리 중...
1500/5669 처리 중...
1550/5669 처리 중...
1600/5669 처리 중...
1650/5669 처리 중...
1700/5669 처리 중...
1750/5669 처리 중...
1800/5669 처리 중...
1850/5669 처리 중...
1900/5669 처리 중...
1950/5669 처리 중...
2000/5669 처리 중...
2050/5669 처리 중...
2100/5669 처리 중...
2150/5669 처리 중...
2200/5669 처리 중...
2250/5669 처리 중...
2300/5669 처리 중...
2350/5669 처리 중...
2400/5669 처리 중...
2450/5669 처리 중...
2500/5669 처리 중...
2550/5669 처리 중...
2600/5669 처리 중...
2650/5669 처리 중...
2700/5669 처리 중...
2750/5669 처리 중...
2800/5669 처리 중.

In [18]:
# 셀 6: 조건별 성능 + confidence 요약
import pandas as pd
from sklearn.metrics import f1_score

df = pd.DataFrame(results)

summary = []
for cname in CONDITION_NAMES:
    preds  = df[f'{cname}_pred'].tolist()
    labels = df['true_label'].tolist()
    
    macro_f1 = f1_score(labels, preds, average='macro', 
                        labels=['support','deny','query','comment'],
                        zero_division=0)
    acc      = df[f'{cname}_correct'].mean()
    avg_conf = df[f'{cname}_conf'].mean()
    avg_ent  = df[f'{cname}_entropy'].mean()
    avg_mar  = df[f'{cname}_margin'].mean()
    
    summary.append({
        'condition':   cname,
        'macro_f1':    round(macro_f1, 4),
        'accuracy':    round(acc, 4),
        'avg_conf':    round(avg_conf, 4),
        'avg_entropy': round(avg_ent, 4),
        'avg_margin':  round(avg_mar, 4),
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

  condition  macro_f1  accuracy  avg_conf  avg_entropy  avg_margin
 reply_only    0.0849    0.1032    0.9028       0.2947      0.8245
     useful    0.0550    0.0834    0.9303       0.2457      0.8800
 irrelevant    0.0519    0.0790    0.9316       0.2398      0.8816
conflicting    0.0530    0.0818    0.9364       0.2264      0.8898
      mixed    0.0536    0.0803    0.9345       0.2348      0.8877
    lexical    0.0692    0.1002    0.9138       0.2686      0.8457


In [19]:
# 셀 7: 틀린 케이스에서 confidence 분포 비교
print("=== 틀렸을 때 평균 confidence ===")
for cname in CONDITION_NAMES:
    wrong = df[df[f'{cname}_correct'] == 0]
    right = df[df[f'{cname}_correct'] == 1]
    print(f"{cname:12s} | 틀림: {wrong[f'{cname}_conf'].mean():.4f} "
          f"| 맞음: {right[f'{cname}_conf'].mean():.4f} "
          f"| 차이: {wrong[f'{cname}_conf'].mean() - right[f'{cname}_conf'].mean():.4f}")

=== 틀렸을 때 평균 confidence ===
reply_only   | 틀림: 0.9104 | 맞음: 0.8368 | 차이: 0.0736
useful       | 틀림: 0.9321 | 맞음: 0.9108 | 차이: 0.0213
irrelevant   | 틀림: 0.9328 | 맞음: 0.9181 | 차이: 0.0146
conflicting  | 틀림: 0.9386 | 맞음: 0.9119 | 차이: 0.0267
mixed        | 틀림: 0.9362 | 맞음: 0.9145 | 차이: 0.0217
lexical      | 틀림: 0.9210 | 맞음: 0.8492 | 차이: 0.0718


In [20]:
# 셀 8: 결과 저장
save_path = "/home/ubuntu/qwen_project/confidence_results_1.5b_ft.json"
df.to_json(save_path, orient='records', force_ascii=False, indent=2)
print(f"저장 완료: {save_path}")

# 요약도 저장
summary_df.to_csv("/home/ubuntu/qwen_project/confidence_summary_1.5b_ft.csv", index=False)
print("요약 저장 완료")

저장 완료: /home/ubuntu/qwen_project/confidence_results_1.5b_ft.json
요약 저장 완료


In [21]:
# 셀 9: 1.5B adversarial 모델 로드
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

ADV_ADAPTER_PATH = "/home/ubuntu/qwen_project/result/qwen_1.5b_adv/final"

# 기존 base_model 재사용 (메모리 절약)
del model
torch.cuda.empty_cache()

base_model_adv = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model_adv, ADV_ADAPTER_PATH)
model.eval()
print("ADV 모델 로드 완료")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 521.67it/s]


ADV 모델 로드 완료


In [22]:
# 셀 10: adv 모델 confidence 측정
results_adv = []
for i, target in enumerate(dev_data):
    if i % 50 == 0:
        print(f"{i}/{len(dev_data)} 처리 중...")

    conds = build_all_conditions(target, threads, id_to_record)
    row = {
        'reply_id': target['reply_id'],
        'true_label': target['label'],
    }
    for cname in CONDITION_NAMES:
        prompt = conds[cname]
        pred, probs, conf, ent, margin = get_confidence_scores(prompt)
        row[f'{cname}_pred']    = pred
        row[f'{cname}_conf']    = conf
        row[f'{cname}_entropy'] = ent
        row[f'{cname}_margin']  = margin
        row[f'{cname}_correct'] = int(pred == target['label'])
    results_adv.append(row)

print(f"\n완료! 총 {len(results_adv)}개")

0/5669 처리 중...
50/5669 처리 중...
100/5669 처리 중...
150/5669 처리 중...
200/5669 처리 중...
250/5669 처리 중...
300/5669 처리 중...
350/5669 처리 중...
400/5669 처리 중...
450/5669 처리 중...
500/5669 처리 중...
550/5669 처리 중...
600/5669 처리 중...
650/5669 처리 중...
700/5669 처리 중...
750/5669 처리 중...
800/5669 처리 중...
850/5669 처리 중...
900/5669 처리 중...
950/5669 처리 중...
1000/5669 처리 중...
1050/5669 처리 중...
1100/5669 처리 중...
1150/5669 처리 중...
1200/5669 처리 중...
1250/5669 처리 중...
1300/5669 처리 중...
1350/5669 처리 중...
1400/5669 처리 중...
1450/5669 처리 중...
1500/5669 처리 중...
1550/5669 처리 중...
1600/5669 처리 중...
1650/5669 처리 중...
1700/5669 처리 중...
1750/5669 처리 중...
1800/5669 처리 중...
1850/5669 처리 중...
1900/5669 처리 중...
1950/5669 처리 중...
2000/5669 처리 중...
2050/5669 처리 중...
2100/5669 처리 중...
2150/5669 처리 중...
2200/5669 처리 중...
2250/5669 처리 중...
2300/5669 처리 중...
2350/5669 처리 중...
2400/5669 처리 중...
2450/5669 처리 중...
2500/5669 처리 중...
2550/5669 처리 중...
2600/5669 처리 중...
2650/5669 처리 중...
2700/5669 처리 중...
2750/5669 처리 중...
2800/5669 처리 중.